# Bài 26: Deploy lên Streamlit Community Cloud
## Biến repo thành một URL công khai — bài cuối!

**Mục tiêu:**
- Chuẩn bị repo cho deploy (Dockerfile, .dockerignore, secrets)
- Deploy miễn phí lên Streamlit Community Cloud → lấy URL
- Cập nhật README với link demo

**Vì sao đây là bước quan trọng nhất cho CV:** recruiter click link là dùng thử ngay trong 10 giây — khác hẳn 'xem code ở repo này'. Một product có URL sống > mười project nằm im trên GitHub.

---
## Phần 1: Lý thuyết

### Streamlit Community Cloud là gì?

Dịch vụ **miễn phí** của Streamlit, deploy app trực tiếp từ **GitHub repo công khai**. Nó cần đúng 3 thứ:
1. `requirements.txt` — để cài thư viện
2. Đường dẫn file chính — `app/main.py`
3. Secrets — API key (dán ở dashboard, KHÔNG commit `.env`)

---

### Secrets: key trên cloud đi đâu?

Không có `.env` trên cloud. Bạn dán key vào phần **Secrets** của app (định dạng TOML):
```toml
GOOGLE_API_KEY = "AIza..."
```
Nhớ bài 25: `config._default_key()` đã đọc `st.secrets["GOOGLE_API_KEY"]` rồi → không cần sửa code, chỉ cần dán key vào dashboard.

> ⚠️ Dùng **key mới** cho cloud (đừng lộ key chính). Và vì đã đổi sang `gemini-flash-latest` ở bài 25, key mới vẫn chạy tốt.

---

### Lưu ý: ổ đĩa trên cloud là tạm thời (ephemeral)

Streamlit Cloud **reset ổ đĩa mỗi lần app ngủ dậy/restart**. Nghĩa là `data/chroma_db` (tài liệu user upload/kéo về) sẽ **mất** sau mỗi lần reboot.

→ May là app của ta **thiết kế kho rỗng từ đầu** (bài 19-23): user tự upload/kéo EDGAR trong phiên dùng. Nên mất data khi reboot **chấp nhận được** cho một demo. (Muốn bền vững thật thì cần vector DB đám mây như Chroma Cloud / Pinecone — ngoài phạm vi bài này.)

---
## Phần 2: Chuẩn bị repo (code)

### 2a. Sửa `Dockerfile` — đang trỏ file cũ `phase4/app.py`

> Streamlit Cloud KHÔNG dùng Dockerfile, nhưng ta sửa cho đúng để ai chạy bằng Docker vẫn được (và cho gọn CV).

Thay toàn bộ bằng:
```dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy package app/ (không copy .env, data — xem .dockerignore)
COPY app/ ./app/

EXPOSE 8501

CMD ["streamlit", "run", "app/main.py", "--server.address", "0.0.0.0", "--server.port", "8501"]
```

### 2b. Sửa `.dockerignore` — loại thư mục `data/` mới

Đổi 2 dòng cũ `phase4/chroma_db/` và `phase4/data/` thành:
```
# Data runtime (mount volume khi run)
data/
```

### 2c. (Tùy chọn) Thêm `requests` vào `requirements.txt`

`edgar.py` dùng `requests`. Nó vốn được kéo theo bởi thư viện khác, nhưng ghi rõ ra cho chắc:
```
requests==2.34.2
```

### 2d. Kiểm tra `.env` KHÔNG bị commit

`.gitignore` phải có `.env` (đã có từ trước). Chạy `git status` — nếu thấy `.env` trong danh sách thì DỪNG, thêm vào `.gitignore` ngay. **Không bao giờ đẩy API key lên GitHub.**

---
## Phần 3: Deploy (các bước trên web)

### Bước 1: Đẩy toàn bộ lên GitHub
```bash
git add .
git commit -m "chore: prep for deploy (Dockerfile, dockerignore)"
git push
```
Repo phải **public** (Settings → General → Danger Zone → Change visibility) để bản free deploy được.

### Bước 2: Tạo app trên Streamlit Cloud
1. Vào **share.streamlit.io** → đăng nhập bằng GitHub
2. **Create app** → **Deploy a public app from GitHub**
3. Điền:
   - Repository: `<username>/Multi-Agent-Research-Assistant`
   - Branch: `main`
   - **Main file path: `app/main.py`**  ← quan trọng

### Bước 3: Dán Secrets
Trước khi bấm Deploy → **Advanced settings** → **Secrets**, dán:
```toml
GOOGLE_API_KEY = "AIza..."
```
(dùng key mới, không phải key trong `.env` của bạn)

### Bước 4: Deploy
Bấm **Deploy**. Lần đầu build ~5-10 phút (cài `sentence-transformers`, `torch`... khá nặng). Khi xong bạn có URL kiểu:
```
https://<tên-app>.streamlit.app
```

### Bước 5: Test URL thật
- Mở link → hỏi 1 câu (chưa upload gì → chỉ có dữ liệu yfinance)
- Kéo 10-Q từ SEC cho 1 ticker → hỏi lại → có citations
- Thử trên điện thoại → chia sẻ được cho bất kỳ ai

> Nếu build lỗi 'out of memory': `sentence-transformers` + `torch` khá nặng so với hạn mức free. Cách giảm: bỏ `ipykernel`, `ddgs` khỏi `requirements.txt` (app không dùng). Nếu vẫn lỗi, đó là giới hạn RAM của bản free — cân nhắc model embedding nhẹ hơn.

---
## Phần 4: Cập nhật README với link demo

Thêm ngay đầu `README.md`, dưới tiêu đề:
```markdown
**🔗 Live demo: https://<tên-app>.streamlit.app**

> Upload báo cáo PDF hoặc gõ 1 ticker để tự kéo 10-Q từ SEC, rồi hỏi đáp có trích dẫn nguồn.
```

Và một mục Deploy ngắn:
```markdown
## Deployment
Deployed on Streamlit Community Cloud. To run your own:
1. Fork this repo
2. Deploy at share.streamlit.io, main file `app/main.py`
3. Add `GOOGLE_API_KEY` in Secrets
```

---

## 🎉 Hoàn thành!

Bạn vừa đi từ *'0 kinh nghiệm LLM'* đến một **product AI đa tác tử deploy công khai**:
- Multi-agent (LangGraph): rút mã → thu thập → phân tích
- RAG có trích dẫn nguồn, kho tài liệu do user tự xây (upload + auto-detect + EDGAR)
- Conversation memory, logging, Docker, API key linh hoạt
- **Một URL để đưa vào CV**

**Checklist cuối:**
- [ ] App chạy trên URL công khai
- [ ] README có link demo + ảnh chụp màn hình
- [ ] Repo sạch (không lộ `.env`, không file rác)
- [ ] Pin repo trên GitHub profile